In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_55_try_0.pickle

In [ ]:
%%cudf.pandas.profile
### cell 56 ###
# Convert the string column to an ordered categorical so that value_counts
# and subsequent arithmetic happen on the GPU, and embed our custom order
# into the category to avoid a separate CPU‐bound reindex.
def count_then_return_percent_68(df, col):
    # create an ordered categorical with our reversed display order
    cat_type = pd.CategoricalDtype(
        categories=responses_in_order_cell68[::-1],
        ordered=True
    )
    s = df[col].astype(cat_type)
    # GPU‐accelerated count over the categorical codes
    counts = s.value_counts(dropna=False, sort=False)
    # avoid a full reduction on the GPU by using the length (O(1))
    total = len(s)
    # compute and round percentages on the GPU
    pct = (counts * 100 / total).round(1)
    # mask out any categories with zero counts so they show up as NaN
    return pct.where(counts != 0)

# question and ordering unchanged
question_of_interest_cell68 = (
    'Of the cloud platforms that you are familiar with, '
    'which has the best developer experience (most enjoyable to use)? - Selected Choice'
)
responses_in_order_cell68 = [
    ' Amazon Web Services (AWS) ', ' Google Cloud Platform (GCP) ',
    ' Microsoft Azure ', ' IBM Cloud / Red Hat ', ' Oracle Cloud ',
    ' VMware Cloud ', ' SAP Cloud ', ' Tencent Cloud ',
    ' Alibaba Cloud ', ' Huawei Cloud '
]

# now this runs entirely on the GPU (no .reindex or .sum() on the CPU)
percentages_cell68 = count_then_return_percent_68(
    responses_df_2022_cell10,
    question_of_interest_cell68
)
percentages_cell68.info()